In [2]:
import pandas as pd
import requests
import json
import time

# Charger la liste générée par le premier fichier
with open("sp500_tickers.json", "r") as f:
    tickers_sp500 = json.load(f)


def get_stocktwits_sentiment(ticker):
    ticker = ticker.upper()
    url = f"https://api.stocktwits.com/api/2/streams/symbol/{ticker}.json"

    # Simulation d'un vrai navigateur moderne pour éviter le blocage 403
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
    }

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        messages = response.json().get("messages", [])
        data = []

        for msg in messages:
            entities = msg.get("entities", {})
            sentiment_obj = entities.get("sentiment") if entities else None
            sentiment_tag = (
                sentiment_obj.get("basic") if sentiment_obj else "Neutre"
            )

            data.append(
                {
                    "ticker": ticker,
                    "text": msg.get("body", ""),
                    "created_at": msg.get("created_at", ""),
                    "sentiment_tag": sentiment_tag,
                }
            )

        return pd.DataFrame(data)
    else:
        print(f"⚠️ Erreur d'accès à l'API (Code {response.status_code})")
        return pd.DataFrame()
    
all_social_data = []
tickers_a_tester = tickers_sp500[:10]  # Prendre les 10 premiers

for ticker in tickers_a_tester:
    print(f"Récupération StockTwits pour : {ticker}")

    df_ticker = get_stocktwits_sentiment(ticker)
    if not df_ticker.empty:
        all_social_data.append(df_ticker)

    time.sleep(2)  # Pause anti-blocage

# 4. Sauvegarder les données récupérées
df_final = pd.concat(all_social_data, ignore_index=True)
df_final.to_csv("stocktwits_sentiment_raw.csv", index=False)
print(f"✅ Terminé ! {len(df_final)} messages sauvegardés.")   
    


"""# Test d'exécution 1 
df = get_stocktwits_sentiment("AAPL")
print(f"Nombre de lignes récupérées : {len(df)}")
print(df.head())"""


Récupération StockTwits pour : MMM
Récupération StockTwits pour : AOS
Récupération StockTwits pour : ABT


KeyboardInterrupt: 